# 01 — The KV cache walkthrough

**Phase 0, exercise 2.** Goal: understand and *measure* the optimization that makes inference tolerable.

In notebook 00 you wrote `greedy_decode_naive`, which reruns the model on the entire growing prefix at every step. By the end of this notebook you will:

- Understand what a KV cache is and why it exists.
- See HuggingFace's `past_key_values` machinery up close.
- Write `greedy_decode_cached` and verify it produces *bit-identical* output to the naive version.
- Benchmark cached vs naive at several sequence lengths and plot the speedup.
- Have a precise answer for the most important vocabulary inflection in inference engineering: **prefill** vs **decode**.

**Compute:** T4 makes the timing curves prettier and lets you push to longer sequences, but CPU works — the benchmark cell below adapts its schedule to whichever device you're on.

## Setup

> Same Colab gotcha as notebook 00: do **not** `pip install torch`. If you hit `partially initialized module 'torch'`, the fix is **Runtime → Restart session** and re-run *without* reinstalling torch.

In [ ]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "EleutherAI/pythia-160m"
MODEL_REVISION = "step143000"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 0

torch.manual_seed(SEED)
print(f"torch {torch.__version__} | device: {DEVICE}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, revision=MODEL_REVISION).to(DEVICE)
model.eval()
print(f"loaded {MODEL_NAME} @ {MODEL_REVISION} | {sum(p.numel() for p in model.parameters()):,} params")

## What is a KV cache?

Recap of where the waste lives, in your own framing from `docs/concepts/01_decoding_basics.md`: in `greedy_decode_naive`, every iteration recomputes the model on the entire growing sequence. The wasteful part isn't running the model on the new token — that's the work we actually need. The wasteful part is **recomputing the K (key) and V (value) projections for every token we already processed**, just so the new token has something to attend to.

A KV cache fixes this by *storing* K and V at every layer for every token already seen. When a new token arrives:

1. Compute its Q, K, V projections (just for the new token).
2. **Append** its K, V to the cache.
3. The new token's Q attends over the **entire cache** of K, V (cheap — one matrix multiply per layer).

Per-step cost drops from re-running the model on `t` positions to running it on **one** position whose attention sees `t` cached positions.

**Memory cost.** The cache is huge. For a model with `L` layers, `H` heads, head dim `d`, and `T` tokens cached, the cache holds `2 × L × H × d × T` floats (× bytes-per-float). For Pythia-160M (`L=12, H=12, d=64`) at 1024 tokens in fp32:

$$ 12 \times 12 \times 64 \times 1024 \times 2 \times 4 \,\text{bytes} \approx 75\,\text{MB} $$

For a 70B model at 32K tokens the cache is tens of GB. That's why "KV cache management" is half the job of an inference engineer.

## What HuggingFace gives us

When you call a HuggingFace causal LM with `use_cache=True`, the output object has a `past_key_values` attribute. It's a per-layer container holding K and V for every token processed so far. On the next call, pass it back as `past_key_values=` and the model will:

- **Append** to it — computing K, V only for the new tokens you passed in.
- **Reuse** the cached values when computing attention.

Two regimes you must learn the names of, because every inference paper uses them:

- **Prefill** — the first call, where you pass the entire prompt at once. The cache gets populated for every prompt position in a single parallel forward pass. *Compute-bound*: lots of work, lots of FLOPs per token.
- **Decode** — every subsequent call, where you pass *only the newest token*. The cache feeds attention over everything before. *Memory-bandwidth-bound*: per-step work is small, but the model weights must be read from VRAM every step.

When inference engineers say "the decode phase is bottlenecked on memory bandwidth," this is what they mean. Modern serving systems (vLLM, TGI, TensorRT-LLM) handle prefill and decode with different schedulers — a fact that only makes sense once you've felt the difference firsthand.

> **API instability note.** The exact Python attributes on the cache object have changed several times across recent `transformers` releases. The decoding loop itself (passing `past_key_values` back through `model(...)`) is stable — only the *inspection* shape churns. The helper in the next cell papers over the variants we've seen in the wild.

In [ ]:
# Run one prefill pass and inspect what the cache actually looks like.
prompt = "The capital of France is"
input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(DEVICE)

with torch.no_grad():
    outputs = model(input_ids, use_cache=True)

pkv = outputs.past_key_values

def get_kv_pairs(cache):
    """Return [(K, V), ...] per layer. Robust to multiple HuggingFace cache APIs."""
    # 1. Most stable across versions: legacy-tuple conversion.
    if hasattr(cache, "to_legacy_cache"):
        try:
            legacy = cache.to_legacy_cache()
            if legacy is not None and len(legacy) > 0:
                return list(legacy)
        except Exception:
            pass
    # 2. Newer API: a .layers list of objects with .keys / .values.
    if hasattr(cache, "layers"):
        return [(layer.keys, layer.values) for layer in cache.layers]
    # 3. Older API: parallel .key_cache / .value_cache lists.
    if hasattr(cache, "key_cache"):
        return list(zip(cache.key_cache, cache.value_cache))
    # 4. Basic sequence protocol: indexing + len.
    try:
        return [(cache[i][0], cache[i][1]) for i in range(len(cache))]
    except Exception:
        pass
    raise RuntimeError(f"unknown cache type: {type(cache).__name__}")

all_KV = get_kv_pairs(pkv)
num_layers = len(all_KV)
K0, V0 = all_KV[0]

print(f"cache class:       {type(pkv).__name__}")
print(f"layers in cache:   {num_layers}")
print(f"layer 0 K shape:   {tuple(K0.shape)}  # [batch, heads, seq_len, head_dim]")
print(f"layer 0 V shape:   {tuple(V0.shape)}")
print(f"prompt length:     {input_ids.shape[1]} tokens")
print(f"cache seq dim:     {K0.shape[2]}  # should equal prompt length")

total_floats = sum(K.numel() + V.numel() for K, V in all_KV)
bytes_per_float = K0.element_size()
print(f"\ncache holds {total_floats:,} floats = {total_floats * bytes_per_float / 1e6:.2f} MB ({K0.dtype})")

## The cached decoding loop

Read carefully — the change from `greedy_decode_naive` is small in lines of code, dramatic in performance:

- On step 0 (**prefill**) we pass the entire prompt and let the model populate the cache.
- On every subsequent step (**decode**) we pass *only the one new token* and rely on the cache for everything else.
- We never re-concatenate growing input ids into the model — we only track them externally for the final decoded string.

In [ ]:
def greedy_decode_cached(model, tokenizer, prompt: str, max_new_tokens: int = 50) -> str:
    """Greedy decode using HuggingFace's KV cache."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    generated = input_ids.clone()
    past_key_values = None
    next_token_id = None

    for step in range(max_new_tokens):
        with torch.no_grad():
            # Step 0 is prefill: pass the full prompt. After that, decode: pass only the new token.
            model_input = input_ids if past_key_values is None else next_token_id
            outputs = model(
                model_input,
                past_key_values=past_key_values,
                use_cache=True,
            )
        past_key_values = outputs.past_key_values

        next_token_logits = outputs.logits[0, -1]
        next_token_id = next_token_logits.argmax(dim=-1).view(1, 1)
        generated = torch.cat([generated, next_token_id], dim=1)

        if next_token_id.item() == tokenizer.eos_token_id:
            break

    return tokenizer.decode(generated[0])

## Sanity check: same output as naive?

Caching is an **optimization, not an approximation**. The cached loop should produce *identical* output to the naive loop on the same prompt with the same sampling strategy (greedy in both cases). If they disagree, something is wrong — most commonly, the cache wasn't being passed back correctly.

(Why must they match exactly? Because the K and V values for a token are deterministic functions of the input and the model weights. Caching them vs recomputing them is the same arithmetic, just done once instead of many times. Floating-point non-determinism *can* cause a tiny mismatch in extreme cases, but for fp32 + greedy it almost never does at this scale.)

In [ ]:
def greedy_decode_naive(model, tokenizer, prompt: str, max_new_tokens: int = 50) -> str:
    """From notebook 00 — kept here so this notebook stands alone."""
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(model.device)
    for _ in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(input_ids)
        next_id = outputs.logits[0, -1].argmax().view(1, 1)
        input_ids = torch.cat([input_ids, next_id], dim=1)
        if next_id.item() == tokenizer.eos_token_id:
            break
    return tokenizer.decode(input_ids[0])


prompt = "The capital of France is"
out_naive = greedy_decode_naive(model, tokenizer, prompt, max_new_tokens=30)
out_cached = greedy_decode_cached(model, tokenizer, prompt, max_new_tokens=30)

print("naive: ", out_naive)
print("cached:", out_cached)
print("identical:", out_naive == out_cached)

## Benchmark

Now the fun part — measure the speedup. Two timing rules to commit to muscle memory:

1. **Warm up first.** The first forward pass after model load includes one-time compilation costs. Throw the first call away.
2. **Synchronize before reading the clock (GPU only).** CUDA kernel launches are asynchronous — `time.perf_counter()` right after a model call on GPU returns *before* the GPU is actually done. `torch.cuda.synchronize()` blocks until the GPU has caught up. On CPU this is a no-op.

> **CPU schedule.** Naive decoding has O(t³) cumulative cost — on CPU it gets painful fast. The cell below uses an *asymmetric* schedule: cached runs further than naive, because cached is cheap per step and naive isn't. The plot will still show the divergence clearly where the two overlap.

In [ ]:
import time

# On GPU we run 3 trials and take the min (CUDA scheduling noise).
# On CPU one trial is enough — variance is low and trials cost real time.
DEFAULT_TRIALS = 3 if DEVICE == "cuda" else 1

def time_decode(decoder_fn, prompt: str, max_new_tokens: int, trials: int = DEFAULT_TRIALS) -> float:
    times = []
    for _ in range(trials):
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        t0 = time.perf_counter()
        _ = decoder_fn(model, tokenizer, prompt, max_new_tokens=max_new_tokens)
        if DEVICE == "cuda":
            torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return min(times)

# Warmup — first call always pays one-time compilation costs.
print("warming up...")
_ = greedy_decode_cached(model, tokenizer, "Hello", max_new_tokens=5)
_ = greedy_decode_naive(model, tokenizer, "Hello", max_new_tokens=5)
print("done\n")

prompt = "Once upon a time"

# Asymmetric schedule: cached is cheap per step, naive is O(t^3) cumulative.
if DEVICE == "cuda":
    cached_lengths = [32, 64, 128, 256]
    naive_lengths  = [32, 64, 128, 256]
else:
    cached_lengths = [16, 32, 64, 96]
    naive_lengths  = [16, 32, 48]

cached_times, naive_times = [], []

for n in cached_lengths:
    t = time_decode(greedy_decode_cached, prompt, n)
    cached_times.append(t)
    print(f"cached  n={n:>3}  {t*1000:>7.0f} ms")
print()
for n in naive_lengths:
    t = time_decode(greedy_decode_naive, prompt, n)
    naive_times.append(t)
    print(f"naive   n={n:>3}  {t*1000:>7.0f} ms")

print()
for ci, n in enumerate(cached_lengths):
    if n in naive_lengths:
        ni = naive_lengths.index(n)
        print(f"speedup at n={n:>3}: {naive_times[ni] / cached_times[ci]:.1f}×")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 1, figsize=(7, 4))
ax.plot(naive_lengths, naive_times, "o-", label="naive (no cache)")
ax.plot(cached_lengths, cached_times, "o-", label="cached")
ax.set_xlabel("tokens generated")
ax.set_ylabel("wall-clock time (s)")
ax.set_title(f"Decoding time vs sequence length ({DEVICE.upper()})")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# Headline number: speedup at the largest length both curves share.
shared = [n for n in cached_lengths if n in naive_lengths]
if shared:
    n = shared[-1]
    ci = cached_lengths.index(n)
    ni = naive_lengths.index(n)
    print(f"at n={n}: naive = {naive_times[ni]:.2f}s | cached = {cached_times[ci]:.2f}s | {naive_times[ni]/cached_times[ci]:.1f}× speedup")

## What you should see

Three patterns to confirm in your plot:

1. **The cached curve grows roughly linearly per step.** Each decode step attends over a growing cache, so per-step cost is `O(t)`. Across `T` total steps, cumulative time is `O(T²)` — but with a small constant.
2. **The naive curve grows quadratically per step**, so cumulative time is `O(T³)`. This is why we cap it sooner on CPU — past ~50 tokens it becomes uncomfortable to wait through.
3. **The speedup ratio grows with sequence length.** KV caching matters *more* the longer you generate — which is why long-context inference would be untenable without it.

If your cached version isn't faster, something's wrong. Sanity check: re-run the inspection cell after a few decode steps and confirm `K0.shape[2]` equals `prompt_len + steps_so_far`. If it's smaller, `past_key_values` isn't being threaded through correctly.

## Reflection questions

Write your answers in `docs/concepts/02_kv_cache.md`.

1. The cached version still gets slower as the sequence grows. Why? (Hint: the cache itself is being read every step.)
2. For a hypothetical 7B model with `L=32` layers, `H=32` heads, `head_dim=128`, and 4096 tokens in context, how big is the KV cache in fp16? Compare to the size of the model weights (~14 GB at fp16). When does the cache start dominating memory?
3. **Prefill** runs the whole prompt in one parallel forward pass; **decode** runs one token per call. Which is more sensitive to memory bandwidth, and why? Which is more sensitive to compute?
4. Sketch a scenario where you'd *want* to drop tokens from the KV cache to save memory. What's the cost of dropping them? (This is the "attention sink" / "sliding window" territory — keep the question in mind for your reading list.)

Question 2 is a calibration check on the memory math from §2.2. Plug into `2 × L × H × head_dim × T × bytes_per_element`. Expected: **~2.0 GB** of KV cache for a single batch at 4K context on a 7B model in fp16. Then ask yourself: cache scales linearly with context length, weights don't — at what `T` does the cache equal the weight memory? That crossover is where every serving system's memory model gets interesting.

## What's next

Phase 0 has one more piece of foundation worth doing before we jump into Phase 1's metric library: **extracting attention weights by hand**, so you can later compute attention entropy and attention concentration on your own terms instead of reading them from a library you don't yet trust. That's notebook 02 — short, ~30 lines of code, builds intuition for what `output_attentions=True` actually returns.

Tell me when you're done with the reflection questions for this notebook and I'll write 02.